# HRS Database Introduction
* Date: 8/27/2026
* Author: Peter Perez
* email: 

## Purpose
The purpose of this document is to serve as an interactive learning resource and technical reference for data analysts, researchers, and SQL users querying the RAND Health and Retirement Study (HRS) relational database in Databricks. It establishes a standardized framework for understanding how longitudinal survey data is structured and accessed within our data platform.

## Objective
The objective of these sample queries is to provide a practical, hands-on learning guide for navigating the relational structure of the database. By transitioning through increasing levels of complexity—from basic data retrieval to advanced analytical techniques like multi-table joins, window functions, and Common Table Expressions (CTEs)—these examples demonstrate how to efficiently extract, aggregate, and analyze survey responses across various waves, cohorts, and individual respondents.

## Database Development Approach

Converting a massive, flat, "wide-format" file like the RAND HRS into a relational database is a classic data engineering design pattern. This process is called Normalization.

In the original flat file, variables use a strict prefix convention (like R1AGEY_E, R2AGEY_E, R3AGEY_E) to represent the exact same question asked across different tracking cycles (Wave 1, Wave 2, Wave 3). By separating these columns out, you unlock cleaner, more efficient, and highly scalable SQL queries.

Here is an overview of the architectural approach you used, written in a clear, scannable format for your documentation.

## Overview: From Flat File to Relational Model
The original RAND HRS dataset is distributed as a single "wide" flat file. While simple to look at in a spreadsheet, it contains thousands of columns that combine the Who (Respondent), the When (Wave), and the What (Survey Sections) into individual variable names.

To make this data easier to access, query, and scale, we transformed this flat architecture into a Star-Schema Relational Model inside Databricks.
========================================================================
                         HRS DATA MODEL DIAGRAM
========================================================================
       +------------------+             +-----------------+
       |  DIM_COHORT      |             |  DIM_WAVE       |
       +------------------+             +-----------------+
       | * cohort_id (PK) |             | * wave_id (PK)  |
       +--------+---------+             +--------+--------+
                |                                |
                | (1)                            | (1)
                v                                |
       +------------------+                      |
       |  HUB_RESPONDENT  |                      |
       +------------------+                      |
       | * resp_id   (PK) |                      |
       |   cohort_id (FK) |                      |
       +--------+---------+                      |
                |                                |
                | (1)                            | (Many)
                v                                v
       +--------------------------------------------------+
       | FACT TABLES (Demographics, Health, Employment...) |
       +--------------------------------------------------+
       | * fact_id       (PK)                             |
       |   resp_id       (FK)                             |
       |   wave_id       (FK)                             |
       |   [Cleaned Survey Variables (e.g., AGEY_E)]      |
       +--------------------------------------------------+

========================================================================
LEGEND:
  * (PK) = Primary Key (The unique identifier for a row in that table)
  * (FK) = Foreign Key (The link used to connect to another table)
  | or v = Relationships (Points from "One" to "Many")
========================================================================

## The 3-Step Transformation Strategy

Our migration strategy relied on three core principles: isolating static characteristics, standardizing timeline intervals, and unpivoting repeating columns into centralized functional buckets.

### 1. Separating the "Dimensions" (The Context)
We extracted variables that do not change over time, or variables that define the structural timeline of the study, and moved them into standalone Dimension Tables:

  * DIM_COHORT: Houses the static birth-year group defining a participant's generation.

  * DIM_WAVE: Houses the chronological survey tracking years.

### 2. Establishing the Central "Hub" (The Entity)

We created a master HUB_RESPONDENT table. This serves as the singular source of truth for who participated in the study. Each person gets one unique ID row, which maps directly back to their immutable generation in the Cohort dimension.

### 3. Transforming and Unpivoting Columns into "Fact Tables" (The Activity)

Instead of keeping 15 different columns for the same question asked across 15 waves, we "unpivoted" the wide data into tall, vertical Fact Tables organized by functional subject areas:
  * FACT_DEMOGRAPHICS
  * FACT_HEALTH
  * FACT_EMPLOYMENT
  * FACT_FINANCIAL
  * FACT_LEAVE_BEHIND


## Tables Created in the relational (other FACT tables will follow) ?

1. DIM_Cohort: A master list of the different age groups (cohorts) participating in the study.

1. DIM_Wave: A master list of the specific years or cycles (waves) when the interviews took place.

1. HUB_Respondent: A master list of the individual people who answered the survey.

1. Fact_Demographics: Background information about the people (such as age, gender, and marital status).

1. Fact_Health: Information regarding the physical and mental health responses of the participants.

1. Fact_Leave-Behind: Special data from paper surveys that participants filled out and mailed back later.

### How the Tables Connect (The Relationships)
To get a complete picture of a person's answers, you will often need to connect these lists. They link together using simple, logical rules:

* **One Cohort, Many Respondents**: Each **Cohort** (age group) contains many different **Respondents** (individual people).

* **One Repondent, Many Records**: Over time, a single **Respondent** will have multiple entries in the **Demographics**, **Health**, and **Leave-Behind** tables because they participate in the survey year after year.

* **One Survey Year (Wave), Many Records**: Each **Wave** (survey year) contains answers from thousands of different people, meaning a single wave connects to many rows in the **Demographics, Health, and Leave-Behind** lists.


NEXT --> Let review Basic SQL Queries is.
[Open Notebook - 0.1.1_SQL_Basics](./0.1.1_SQL_Basics.ipynb)


